An agent:

- Has instructions
- Has tools
- Has memory

In [2]:
from anthropic import Anthropic
client = Anthropic()

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
messages=[
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=messages
)

response.content[0].text

"I'd be happy to help! However, I don't have context about which course you're referring to. Could you provide more details, such as:\n\n- **Course name** - What's it called?\n- **Where you found it** - Is it on Udemy, Coursera, a university, or elsewhere?\n- **Current status** - Is enrollment still open?\n\nOnce you share those details, I can give you better guidance on how to join!"

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search_tool = {
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"]
    }
}

In [7]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=messages,
    tools=[search_tool]
)

In [8]:
len(response.content)

2

In [9]:
response.content[1]

ToolUseBlock(id='toolu_01LM6gTjPQWEdJ69b4TF1oMe', caller=DirectCaller(type='direct'), input={'query': 'how to join enroll register course'}, name='search', type='tool_use')

In [10]:
tool_use = response.content[1]

In [11]:
tool_use

ToolUseBlock(id='toolu_01LM6gTjPQWEdJ69b4TF1oMe', caller=DirectCaller(type='direct'), input={'query': 'how to join enroll register course'}, name='search', type='tool_use')

In [12]:
args = tool_use.input
print(args)

{'query': 'how to join enroll register course'}


In [13]:
results = search(args['query'])
print(results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '04919992b3', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How should I start the course and follow the weekly workflow?', 'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson video

What we've done so far:

1. Make a call to the LLM <-- (first call)
2. LL decided to invoke search with 'params'
3. Then we invoked the search and got results back
4. Now we'll send the results back to the LLM <-- (another call)

In [14]:
#turning results into json
import json

results_json = json.dumps(results, indent = 2)
print(results_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

In [15]:
tool_result = {
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": results_json
        }
    ]
}


In [16]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[
        *messages,                          # original user message
        {"role": "assistant", "content": response.content},  # Claude's tool call response
        tool_result                # your tool result
    ],
    tools=[search_tool]
)

In [17]:
print(response.content[0].text)

Great question! **Yes, you can join the course!** Here are the key details:

✅ **You can join anytime** - The videos and course materials are available, so you can start learning whenever you want.

📋 **No registration needed** - You don't need a confirmation email or to be on a registered list. You can simply start learning and submitting homework right away.

🏆 **Certificate requirements** - If you want to receive a certificate, there are a couple of important things to know:
- You need to **submit your project before the submission deadline** closes
- You can only get a certificate if you complete the course with a "live" cohort (not in self-paced mode)
- After submitting your project, you'll need to peer-review 3 capstone projects from other students (this happens when the course is actively running)

📅 **Next course offering** - The course will be offered again in Summer 2027.

To get started:
1. Check out the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)


Now the

5. LLM processes the results
6. LLM gives the answer

In [18]:
usage = response.usage

In [19]:
usage.input_tokens, usage.output_tokens

(1438, 323)

Total Process:

1. Make a call to the LLM <-- (first call)
2. LL decided to invoke search with 'params'
3. Then we invoked the search and got results back
4. Now we'll send the results back to the LLM <-- (another call)
5. LLM processes the results
6. LLM gives the answer

But what if the LLM needs to make another tool call and the process looks like??

1. Make a call to the LLM <-- (first call)
2. LL decided to invoke search with 'params'
3. Then we invoked the search and got results back
4. Now we'll send the results back to the LLM <-- (another call)
5. LLM processes the results
6. LLM decides to make another tool call 
7. Send another API request (call)
8. LLM Processes and gives the answer

^This is what the agentic loop is for

In [20]:
def make_call(tool_use):
    args = tool_use.input  # already a dict, no json.loads needed

    if tool_use.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": result_json
            }
        ]
    }

In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

messages=[
    {"role": "user", "content": question}
]

In [22]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=instructions,
    messages=messages,
    tools=[search_tool]
)

In [23]:
response.content[0].text

"I'd be happy to help you find information about joining the course! Let me search the FAQ for enrollment and registration information."

In [24]:
messages.extend(response.content)

for item in response.content:
    
    if item.type == "tool_use":
        # process function call 
        print(f"Tool use detected: {item.name} with input {item.input}")
        call_output=make_call(item)
        messages.append(call_output)

    elif item.type == "message":
        print('ASSISTANT:')
        print(item.content[0].text)

Tool use detected: search with input {'query': 'join course enrollment registration'}
Tool use detected: search with input {'query': 'how to enroll sign up'}
Tool use detected: search with input {'query': 'course admission requirements'}


In [25]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 TextBlock(citations=None, text="I'd be happy to help you find information about joining the course! Let me search the FAQ for enrollment and registration information.", type='text'),
 ToolUseBlock(id='toolu_01PEksT85yTWnVHUm1nr5KfC', caller=DirectCaller(type='direct'), input={'query': 'join course enrollment registration'}, name='search', type='tool_use'),
 ToolUseBlock(id='toolu_01XCwRwutpfSkqg2sbo1dNY5', caller=DirectCaller(type='direct'), input={'query': 'how to enroll sign up'}, name='search', type='tool_use'),
 ToolUseBlock(id='toolu_01JVPM6ryevAUSn13vv4LSZG', caller=DirectCaller(type='direct'), input={'query': 'course admission requirements'}, name='search', type='tool_use'),
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01PEksT85yTWnVHUm1nr5KfC',
    'content': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Qu

In [26]:
messages=[
    {"role": "user", "content": question}
]


it = 1

while True:

    print(f'iteration #{it}')
    has_tool_use = False

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system=instructions,
        messages=messages,
        tools=[search_tool]
    )

    messages.append({"role": "assistant", "content": response.content})

    has_tool_use = False

    for item in response.content:
        
        if item.type == "tool_use":
            # process function call 
            print(f"Tool use detected: {item.name} with input {item.input}")
            call_output=make_call(item)
            messages.append(call_output)
            has_tool_use = True

        elif item.type == "text":
            print('ASSISTANT:')
            print(item.text)

    it = it + 1

    if has_tool_use == False:
        break


response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=instructions,
    messages=messages,
    tools=[search_tool]
)

messages.append({"role": "assistant", "content": response.content})

for item in response.content:
    
    if item.type == "tool_use":
        # process function call 
        print(f"Tool use detected: {item.name} with input {item.input}")
        call_output=make_call(item)
        messages.append(call_output)

    elif item.type == "text":
        print('ASSISTANT:')
        print(item.text)

iteration #1
ASSISTANT:
I'd be happy to help you with information about joining the course! Let me search the FAQ for details about enrollment and registration.
Tool use detected: search with input {'query': 'join course enrollment registration'}
Tool use detected: search with input {'query': 'can I join late enroll new students'}
iteration #2
ASSISTANT:
Great! I found the answer to your question. Here's what you need to know:

**Yes, you can join the course!** 

According to the course FAQ:
- You can join even if you just discovered it
- **However, if you want to receive a certificate**, you'll need to submit your project while the course is still accepting submissions

Also, some helpful additional details:
- You don't even need to formally register to participate - registration is just to gauge interest before the start date. You can start learning and submitting homework anytime the form is open.
- You can start the course whenever you want - the videos and materials are available.

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

messages=[
    {"role": "user", "content": question}
]

In [28]:
def agent_loop(question, instructions, model='claude-haiku-4-5-20251001') -> str:
    
    messages = [{"role": "user", "content": question}]
    it = 1
    last_answer = ""


    while True:

        print(f'iteration #{it}')
        has_tool_use = False

        response = client.messages.create(
            model=model,
            max_tokens=1024,
            system=instructions,
            messages=messages,
            tools=[search_tool]
        )

        messages.append({"role": "assistant", "content": response.content})

        has_tool_use = False

        for item in response.content:
            
            if item.type == "tool_use":
                # process function call 
                print(f"Tool use detected: {item.name} with input {item.input}")
                call_output=make_call(item)
                messages.append(call_output)
                has_tool_use = True

            elif item.type == "text":
                print('ASSISTANT:')
                print(item.text)
                last_answer = item.text

        it += 1

        if not has_tool_use:
            break
        
    return last_answer


In [29]:
agent_loop("How do I run ollama locally?", instructions)

iteration #1
ASSISTANT:
I'll search the FAQ database for information about running Ollama locally.
Tool use detected: search with input {'query': 'run ollama locally'}
Tool use detected: search with input {'query': 'ollama setup installation'}
Tool use detected: search with input {'query': 'ollama configuration local'}
iteration #2
ASSISTANT:
Great! I found comprehensive information about running Ollama locally. Here's how to do it:

## Installation Steps

First, **install Ollama** by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:

- **macOS**: Download the `.pkg` and install it.
- **Windows**: Download the `.msi` and install it.
- **Linux**: Run the following command in the terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

## Running Ollama

Once installed, open a terminal and type:

```bash
ollama run llama3
```

This command will:
- Download the LLaMA 3 model (~4GB)
- Start the model locally
- Open a c

'Great! I found comprehensive information about running Ollama locally. Here\'s how to do it:\n\n## Installation Steps\n\nFirst, **install Ollama** by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\n## Running Ollama\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n- Download the LLaMA 3 model (~4GB)\n- Start the model locally\n- Open a chat-like interface where you can type questions\n\n## Testing the Local Server\n\nTo verify that the Ollama local server is running, execute:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n```json\n{"models": [...]}\n```\n\n## Using Ollama with Python\n\n1. Install the Pytho

Now we're going to add a framework he made (ToyAIkit) that does pretty much the same agentic loop we did above.

In [30]:
!uv add toyaikit

Resolved 127 packages in 13ms
Checked 123 packages in 111ms


In [31]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [32]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [33]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [34]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [35]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [36]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

I can't use the above because I started this with Anthropic AI, and he built the tool with Open AI (I don't have and will not pay another $5 to use OpenAI's API)